<a href="https://colab.research.google.com/github/victoriabrowwn/lab-4-llm-decision-support/blob/main/lab_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 4: LLMs and Prompt Engineering for Decision Support

**Duration:** 2 weeks [30 Jul - 13 Aug, 2026]
**Due Date:** 13th August, 2026
**Format:** Jupyter Notebook / Google Colab + external APIs + GitHub version control
**Grading:** This is a graded lab.

**Student Name:** Victoria Brown Afari
**Student ID:** 94692028

---

### Objective

In the previous labs you *trained* models. In this lab you will *use* a model that someone
else spent millions of dollars training — a **Large Language Model (LLM)** — and learn that
getting good results out of one is an engineering discipline of its own: **prompt
engineering**.

You will build a **decision support system for a microfinance loan officer**. Given a pile of
free-text loan application letters, your system will:

1. **Summarize** each application into a short, factual brief,
2. **Extract** specific structured data points (JSON) that a downstream system could store,
3. Produce a **decision-support recommendation** — while keeping the human firmly in the loop.

Just as importantly, you will **evaluate** the LLM's output for quality, reliability, and
appropriateness: Does it hallucinate? Is it consistent across runs? Should it be trusted to
make the final call?

---

### Choosing an API provider

You need an LLM API with a **free tier**. Recommended options (pick ONE):

| Provider | Free tier | Notes |
|---|---|---|
| **Groq** (recommended) | Yes, generous | OpenAI-compatible API, very fast, open models (Llama) |
| **Google Gemini** | Yes | `google-generativeai` package |
| **Hugging Face Inference API** | Yes, limited | Many open models |
| OpenAI / Anthropic | Paid | Fine if you already have credits |

The notebook's example code uses the **OpenAI-compatible chat format** (works with Groq and
OpenAI directly; Gemini users adapt the call in one place). Everything else in the lab is
provider-agnostic.

---
### Part 0: Repository and API-key setup

1. Create a **public** repository named `lab-4-llm-decision-support` and save this notebook
   inside it.
2. Sign up with your chosen provider and create an **API key**.
3. **NEVER hard-code or commit your API key.** This is a graded requirement.
   - Locally: put it in a `.env` file and add `.env` to `.gitignore`.
   - Colab: use the Secrets panel (key icon) and read it with `google.colab.userdata`.
4. Add a `requirements.txt`: `openai python-dotenv pandas matplotlib`.
5. Commit and push after **each Part** — we will check for incremental commits.

> **A leaked key in your commit history = resubmission + penalty.** Keys can be scraped from
> public repos within minutes.

In [14]:
# API-key setup — DO NOT hard-code your key in this cell.

import os

# --- Local (with a .env file) ---
# from dotenv import load_dotenv
# load_dotenv()
# API_KEY = os.environ["GROQ_API_KEY"]

# --- Google Colab (Secrets panel) ---
from google.colab import userdata
API_KEY = userdata.get("GROQ_API_KEY")

# TODO: set API_KEY using ONE of the methods above.

# OpenAI-compatible client (works for Groq and OpenAI; Gemini users see their docs):
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",   # remove this line if using OpenAI itself
)
MODEL = "llama-3.3-70b-versatile"                # or your provider's model name

print("Client ready.")

Client ready.


---
# Section 1 — Talking to an LLM Programmatically

Before building anything, understand the anatomy of an API call: **messages and roles**
(`system`, `user`, `assistant`), and the **generation parameters** (`temperature`,
`max_tokens`).

### Part 1.1 — Your first API call

In [15]:
import os
# TODO: Write a helper function you will reuse for the WHOLE lab:
#
def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
            temperature=0.7, max_tokens=500):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response # Return the full response object
#
# TODO: Call it once with a simple question and print the answer.
response = ask_llm("What is the capital of France?") # Store the full response
# Access the content
print(response.choices[0].message.content)
# TODO: Print response.usage as well — how many tokens did your call consume?
print(response.usage)

The capital of France is Paris.
CompletionUsage(completion_tokens=8, prompt_tokens=48, total_tokens=56, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.054057726, prompt_time=0.004462526, completion_time=0.011921373, total_time=0.016383899)


**Student Reasoning — Anatomy of a call**
*1. What is the difference between the `system` and `user` roles? Give an example of
something that belongs in each.*
*2. What is a token, roughly? Why do API providers bill per token rather than per request?*

> **Answer:**
1. The system roles sets the assistant's behaviour for the entire conversation(eg. "You are an assistant to a microfinance loan officer. Be factual and neutral. Never invent details not present in the letter). The user role carries the sepcific task or content for that turn(eg. "Summarise this loan application).

2. A token roughly is a chunck of text. API providers bill per token because token count is a direct proxy for the compute the model actually did. Billing per request would either massively overcharge a one-word question or undercharge a 2000 word request. Token based billing tiescost to actual work done.

### Part 1.2 — Temperature: the randomness dial

In [16]:
# TODO: Ask the SAME question 5 times at temperature=0.0 and 5 times at temperature=1.2.
#   A good test question: "Suggest a name for a savings product for market traders in Accra."
for i in range(5):
  response = ask_llm("Suggest a name for a savings product for market traders in Accra.", temperature=0.0)
  print(response.choices[0].message.content)

for i in range(5):
  response = ask_llm("Suggest a name for a savings product for market traders in Accra.", temperature=1.2)
  print(response.choices[0].message.content)
# TODO: Print all 10 answers, grouped by temperature.

Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **Trader's Treasure**: This name emphasizes the idea of saving and accumulating wealth.
3. **Sika Kokoo**: "Sika" means "money" in the Akan language, and "Kokoo" means "gather" or "collect". This name could appeal to market traders who want to gather and save their earnings.
4. **Market Mobi**: This name incorporates "mobi", short for mobile, to suggest a convenient and accessible savings product.
5. **Adanfo Save**: "Adanfo" means "friends" or "partners" in the Akan language, implying a sense of community and mutual support among market traders.
6. **Kae Dwa**: "Kae Dwa" means "good fortune" or "prosperity" in the Ga language, which could be an attractive name for a savings product.
7. **Traders' Trust**: This name emphasizes the idea of trust and reliability, which is essential for a savings pr

**Student Reasoning — Temperature**
*What did you observe at each temperature? For the loan decision-support system you are about
to build, which temperature regime is appropriate, and why?*

> **Answer:**
1. At temperature=0.0, the suggested product name were large;y the same core set repeated with only minor wording differences run ro run, the low temperature is pushing the model toward its most probable token each time, so outputs cluster tightly. At temperature = 1.2, the outputs were visibly more varied and inventive, new name candidates apperared each run.

For the loan decision support system, low temperature is the right regime. Because summarizing and extracting facts from a loan letter, and supoorting a lending decision, needs consistency and reproducubility.

---
# Section 2 — The Dataset: Loan Application Letters

Run the next cell to load **six loan application letters** submitted to a (fictional)
microfinance institution in Ghana, plus **gold-standard extraction labels** for three of them
(you will use these for evaluation in Section 4).

Read at least two letters fully before moving on — you cannot engineer prompts for text you
have not read.

In [17]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.


---
# Section 3 — Prompt Engineering for the Decision Support System

You will now build the three components of the system, iterating on your prompts as you go.
**Keep every major prompt version** — Section 3.4 asks you to commit your prompt templates
and document how they evolved.

### Part 3.1 — Component 1: Summarization
Turn a rambling letter into a 3-4 sentence factual brief a busy loan officer can scan.

In [18]:
# TODO: Write SUMMARY_PROMPT_V1 — your first, naive attempt (e.g. just "Summarize this:").
#   Run it on L002 and L006. Read the output critically.
def SUMMARY_PROMPT_V1(letter):
  return ask_llm(f"Summarize this: {letter}")


print("Summary V1 results")
print(SUMMARY_PROMPT_V1(LETTERS["L002"]).choices[0].message.content)
print(SUMMARY_PROMPT_V1(LETTERS["L006"]).choices[0].message.content)



# TODO: Now write SUMMARY_PROMPT_V2 as a proper template with:
#   - a system prompt giving the LLM a ROLE (e.g. "You are an assistant to a microfinance
#     loan officer...") and constraints (factual, neutral, no invented details, 3-4 sentences)
#   - a user prompt template like: f"Summarize this loan application:\n\n{letter_text}"
#   Run V2 on the same two letters at temperature=0.

print(" ")
print(" ")

def SUMMARY_PROMPT_V2(letter):
  return ask_llm(user_prompt=f"Summarize this loan application: {letter}", system_prompt = "You are an assistant to a microfinance loan officer", temperature=0)

print("Summary V2 results")
print(SUMMARY_PROMPT_V2(LETTERS["L002"]).choices[0].message.content)
print(SUMMARY_PROMPT_V2(LETTERS["L006"]).choices[0].message.content)

# TODO: Compare V1 vs V2 outputs side by side. Keep both prompt versions in this notebook.

Summary V1 results
Kwame Boateng, a commercial driver in Kumasi, is urgently seeking a loan of GHS 25,000 to repair his vehicle's engine and pay off personal debts. He's experiencing a slow business period but is optimistic it will improve after the festive season and promises to repay the loan as soon as possible, despite not having collateral.
Kofi, a 22-year-old, is seeking a loan of GHS 50,000 to start three businesses: a car washing service, a provision shop, and a phone import business from Dubai. He has no prior experience, but claims to be "business-minded" based on his friends' opinions. He promises to repay the loan in one year, once his businesses are successful, but has no collateral to offer, relying on his personal trustworthiness.
 
 
Summary V2 results
Here is a summary of the loan application:

**Applicant:** Kwame Boateng
**Occupation:** Commercial driver (trotro) in Kumasi
**Loan Amount:** GHS 25,000
**Purpose:** To repair his trotro engine and settle personal debts


**Student Reasoning — Summarization prompts**
*1. What concrete problems did V1's output have that V2 fixed? Quote examples.*
*2. Why is "no invented details" an essential instruction in this application? What is this
failure mode called in the LLM literature?*

> **Answer:**
1. V1 had no imposed structure, so its output was a dense paragraph a busy officer has to read in full to find the relevant facts. It also slipped into sublty inaccurate framing; for L006 it wrote that Kofi "offers his trusthworthiness as collateral", but the letter explicitly says "No collateral". V2 fixed the scanability problem by usin a consistent labeled format(Applicant/Occupation/Loan Amount/Purpose/Repayment Plan/Collateral), which is faster to learn.

2. Could materially change a real financial decision, leading to an unfair denial, an unwarranted approval, or exposing the institution to regulatory/reputational risk if an officer acts on a flase "fact" the system asserted confidently. This failure mode is called hallucination.

### Part 3.2 — Component 2: Structured extraction (JSON)
Downstream software cannot read prose. Extract the fields in `GOLD` as strict JSON.

In [19]:
# TODO: Write EXTRACT_PROMPT — a template that instructs the model to return ONLY a JSON
#   object with EXACTLY these keys:
#     applicant_name (string), amount_ghs (number), purpose (string),
#     monthly_profit_ghs (number or null), has_collateral_or_guarantor (boolean),
#     repayment_months (number or null)
#   Techniques to use:
#     - explicit schema in the prompt
#     - ONE worked example (few-shot) using a letter you write yourself (not from LETTERS!)
#     - "If a field is not stated in the letter, use null. Do not guess."
#     - temperature=0

import json
import pandas as pd
import re # Import regex for stripping fences


def EXTRACT_PROMPT(letter_text):
  # Few_shot example letter
  example_letter = """Dear Sir,
  My name is Victoria Brown Afari and I need GHS 4000 for my small nail tech business.
  I will use it to buy nail equipmemts. The profit I make each month is about GHS 1500.
  I have a guarantor. I can repay in 10 months.
  """

  # Expected JSON for the few-shot example
  example_json = {
      "applicant_name": "Victoria Brown Afari",
      "amount_ghs": 4000,
      "purpose": "small nail tech business",
      "monthly_profit_ghs": 1500,
      "has_collateral_or_guarantor": True,
      "repayment_months": 10
  }

  system_prompt = (
      "You are a highly efficient data extraction bot. Your sole task is to extract specific information "
      "from loan application letters and return it as a JSON object. Adhere strictly to the following schema:\n"
      "- `applicant_name`: (string) The full name of the applicant.\n"
      "- `amount_ghs`: (number) The loan amount requested in Ghanaian Cedis.\n"
      "- `purpose`: (string) A brief description of what the loan is for.\n"
      "- `monthly_profit_ghs`: (number or null) The stated monthly profit of the applicant's business, if mentioned. Use null if not specified.\n"
      "- `has_collateral_or_guarantor`: (boolean) True if collateral or a guarantor is mentioned, False otherwise.\n"
      "- `repayment_months`: (number or null) The proposed repayment period in months, if mentioned. Use null if not specified.\n"
      "If a field is not stated in the letter, use null. Do not guess or invent details. "
      "Respond ONLY with the JSON object. Do not include any other text or formatting outside the JSON."
  )

  user_prompt = (
      f"Extract the following fields from the loan application letter and return them as a JSON object:\n"
      f"Letter:\n{example_letter}\n"
      f"JSON: {json.dumps(example_json)}\n\n"
      f"Letter:\n{letter_text}\n"
      f"JSON:"
  )

  return system_prompt, user_prompt

# TODO: Write extract_fields(letter_text) that calls the LLM, strips any ```json fences,
#   json.loads() the result, and returns a dict. Handle parse failures gracefully
#   (return None and print a warning).
def extract_fields(letter_text):
  system_prompt, user_prompt = EXTRACT_PROMPT(letter_text)
  try:
    llm_response = ask_llm(
        user_prompt=user_prompt,
        system_prompt=system_prompt,
        temperature=0.0 # Strict extraction, so temperature=0
    )

    # The ask_llm function returns the full response object, so we need to access the content
    response_content = llm_response.choices[0].message.content

    # Strip any ```json fences or other extraneous text from the response
    # Use a regex to find content between ```json and ```, or just try to parse directly
    match = re.search(r'```json\n(.*?)```', response_content, re.DOTALL)
    if match:
        json_string = match.group(1).strip()
    else:
        # If no fences, assume the response content IS the JSON string
        json_string = response_content.strip()

    extracted_data = json.loads(json_string)
    return extracted_data
  except json.JSONDecodeError as e:
    print(f"Warning: Could not parse JSON for letter: {letter_text[:50]}...")
    print(f"Error: {e}")
    print(f"Raw LLM response: {response_content}")
    return None
  except Exception as e:
    print(f"An unexpected error occurred during extraction for letter: {letter_text[:50]}...")
    print(f"Error: {e}")
    return None

# TODO: Run it on ALL SIX letters; collect results into a pandas DataFrame (one row per
#   letter) and display it.

extracted_results = []
for letter_id, letter_text in LETTERS.items():
  print(f"Processing {letter_id}...")
  data = extract_fields(letter_text)
  if data:
    data['letter_id'] = letter_id # Add letter_id for identification
    extracted_results.append(data)

df_extracted = pd.DataFrame(extracted_results)
# Reorder columns to have letter_id first
df_extracted = df_extracted[['letter_id'] + [col for col in df_extracted.columns if col != 'letter_id']]

print("\nExtracted Data DataFrame:")
display(df_extracted)

Processing L001...
Processing L002...
Processing L003...
Processing L004...
Processing L005...
Processing L006...

Extracted Data DataFrame:


,letter_id,applicant_name,amount_ghs,purpose,monthly_profit_ghs,has_collateral_or_guarantor,repayment_months
0,L001,Akosua Mensah,8000,buy a deep freezer and expand into frozen foods,900.0,True,20.0
1,L002,Kwame Boateng,25000,repair trotro engine and settle personal debts,NaN,False,NaN
2,L003,Efua Darko,15000,purchase two industrial sewing machines and fa...,2800.0,True,15.0
3,L004,Yaw Owusu,12000,poultry farm,1500.0,True,18.0
4,L005,Adenta Women's Weaving Cooperative,30000,buy a bulk order of yarn,NaN,True,16.0
5,L006,Kofi,50000,"start a car washing business, a provision shop...",NaN,False,12.0


**Student Reasoning — Structured extraction**
*1. Why must the few-shot example NOT come from the six letters you are processing?*
*2. Why "use null, do not guess" — what did the model do without that instruction?*
*3. Why is temperature=0 the right choice for extraction but arguably not for creative tasks?*

> **Answer:**
1. Because it will leak the "answer key" into the prompt, inflating the measured accuracy and telling nothing about how well the extractor generalizes to new letters it has not seen a matching example for.

2. It tends to fill in every field somehow rather than admit absence. Explicitly instructing null for missing data converts "I dont't know" into an honest, checkable signal instead of a silent guess dressed up as a fact.

3. Because the model is to pick its single most probable output every time so results are reproducible and consistent accross repeated runs on the same input. Creative tasks have no single "correct" answer, diversity and novelty are actually desirable there, which is what higher temperature buys.

### Part 3.3 — Component 3: The decision-support brief
Combine everything: for each letter, produce a recommendation brief for the loan officer —
strengths, risks, missing information, and a suggested next step. The system must
**support** the decision, not **make** it.

In [24]:
# TODO: Write BRIEF_PROMPT — it receives the letter AND your extracted JSON, and must output:
#     1. Strengths (bullet points, grounded in the letter)
#     2. Risks / red flags (bullet points)
#     3. Missing information the officer should request
#     4. Suggested next step (e.g. "invite for interview", "request documents",
#        "flag for senior review") — NOT "approve" or "reject".
#   Give the model an explicit instruction that final decisions are made by humans.
def BRIEF_PROMPT(letter_text, extracted_data):
  system_prompt = (
      "You are an assistant to a microfinance loan officer. Your task is to provide a decision support brief "
      "for a loan application. Your brief should be factual, neutral, and based solely on the provided letter "
      "and extracted data. Do not make any final approval or rejection decisions; instead, suggest concrete "
      "next steps for the loan officer. Structure your response as follows:\n\n"
      "**Strengths:**\n- [Bullet point 1]\n- [Bullet point 2]\n...\n"
      "**Risks/Red Flags:**\n- [Bullet point 1]\n- [Bullet point 2]\n...\n"
      "**Missing Information:**\n- [Missing piece 1]\n- [Missing piece 2]\n...\n"
      "**Suggested Next Step:** [e.g., 'Invite for interview', 'Request sales records', 'Flag for senior review']"
  )

  user_prompt = (
      f"Please generate a decision support brief for the following loan application letter and extracted data.\n\n"
      f"**Loan Application Letter:**\n{letter_text}\n\n"
      f"**Extracted Data (JSON):**\n{json.dumps(extracted_data, indent=2)}\n\n"
      "Generate the brief following the specified format. Remember, do not make approval or rejection decisions."
  )

  response = client.chat.completions.create(
      model=MODEL,
      messages=[
          {"role": "system", "content": system_prompt},
          {"role": "user", "content": user_prompt},
      ],
      temperature=0.0,
      max_tokens=500
  )
  return response.choices[0].message.content



# TODO: Generate briefs for ALL SIX letters. Print the briefs for L001, L002, and L006 —
#   three very different applications.
all_briefs = {}
for letter_id, letter_text in LETTERS.items():
  extracted = extract_fields(letter_text)
  if extracted:
    brief = BRIEF_PROMPT(letter_text, extracted)
    all_briefs[letter_id] = brief
  else:
    all_briefs[letter_id] = "Could not extract data for this letter to generate a brief."

print("\n--- Brief for L001 ---")
print(all_briefs["L001"])

print("\n--- Brief for L002 ---")
print(all_briefs["L002"])

print("\n--- Brief for L006 ---")
print(all_briefs["L006"])

print("\n--- Brief for L003 ---")
print(all_briefs["L003"])


--- Brief for L001 ---
**Strengths:**
- The applicant, Akosua Mensah, has a long-standing business experience of 12 years selling provisions at Makola Market.
- She has a stable monthly profit of GHS 900, which indicates a consistent income stream.
- Akosua has demonstrated a savings habit through the susu scheme, accumulating GHS 2,500 over two years without missing any contributions.
- She has a guarantor, her sister, who is a teacher, potentially providing an additional layer of security for the loan.
- The applicant has proposed a repayment plan of GHS 450 monthly over 20 months, which is roughly half of her monthly profit, suggesting a manageable repayment schedule.

**Risks/Red Flags:**
- The loan amount of GHS 8,000 is significant compared to the applicant's monthly profit and savings, which may pose a risk if her business does not expand as planned.
- Expanding into frozen foods with a deep freezer may require additional expenses and skills, potentially impacting the applicant

**Student Reasoning — Decision support**
*1. Compare the briefs for L003 (strong application) and L006 (weak application). Did the
system identify the right strengths and red flags in each?*
*2. Why did we forbid the model from outputting "approve"/"reject"? Give one practical and
one ethical reason.*

> **Answer:**
1. For L006, the system correctly read a weak application as weak. The system had to stetch to find strengths, "enthusiasm and energy" and "diversity of busniness ventures" are not real evidece of creditworthiness, and it is telling that these are the best it could produce. Its risks are structural: no experience in any of the three proposed businesses, no collateral guarantor, a repayment plan based only on hoped-for success, and Kofi's "business-minded" self assessment correctly called out as subjective and unsupported.
For L003, the strengths lsited are concrete and verifiable. The risks it flagged are proportionate, not fundamemntal, it questioned whether the GHS1,100 monthly repayment leaves enough buffer against the GHS2,800 profit, and noted reliance on seasonal revenue.

2. A practical reason is that it keeps a human loan officer accountable for the final credit decision and preserves the audit trail. An imperfect system should not be the final word on approval. An ethical reason is that it keeps meaningful human judgment in high-stakes, life-affecting decision, guarding against the system's biases being converted directly into an automated rejection with no human check.

### Part 3.4 — Commit your prompt templates
Prompts ARE code. Save your final `SUMMARY_PROMPT`, `EXTRACT_PROMPT`, and `BRIEF_PROMPT` into
a separate file `prompts.py` (or `prompts.md`) in your repository and commit it with a
message describing how the prompts evolved. Paste your commit hash below.

> **Commit hash:** [paste here]

---
# Section 4 — Evaluation: Quality, Reliability, Appropriateness

An impressive demo is not a trustworthy system. Now measure it.

### Part 4.1 — Extraction accuracy against gold labels

In [21]:
# TODO: For the three letters in GOLD, compare your extracted DataFrame to the gold values
#   field by field. Compute per-field accuracy across the three letters
#   (name matching can be case-insensitive; numbers must match exactly).

# TODO: Display a small table: rows = fields, columns = L001 / L003 / L006 / accuracy.

# Compare df_extracted (from Part 3.2) to GOLD field by field for L001, L003, L006.

def fields_match(field, predicted, gold):
    """Return True if predicted value matches gold value for this field."""
    # Treat missing/NaN as None for comparison
    if pd.isna(predicted):
        predicted = None
    if predicted is None and gold is None:
        return True
    if predicted is None or gold is None:
        return False

    if field == "applicant_name":
        # case-insensitive string match
        return str(predicted).strip().lower() == str(gold).strip().lower()
    elif field in ("amount_ghs", "monthly_profit_ghs", "repayment_months"):
        # numbers must match exactly
        try:
            return float(predicted) == float(gold)
        except (ValueError, TypeError):
            return False
    elif field == "has_collateral_or_guarantor":
        return bool(predicted) == bool(gold)
    elif field == "purpose":
        # free text -> case-insensitive exact match (strict; note this in your write-up)
        return str(predicted).strip().lower() == str(gold).strip().lower()
    else:
        return predicted == gold

gold_letter_ids = list(GOLD.keys())  # ["L001", "L003", "L006"]
fields = ["applicant_name", "amount_ghs", "purpose",
          "monthly_profit_ghs", "has_collateral_or_guarantor", "repayment_months"]

results_table = {field: {} for field in fields}

for letter_id in gold_letter_ids:
    row = df_extracted[df_extracted["letter_id"] == letter_id]
    if row.empty:
        print(f"Warning: {letter_id} not found in df_extracted.")
        continue
    row = row.iloc[0]
    gold_row = GOLD[letter_id]

    for field in fields:
        predicted_value = row[field]
        gold_value = gold_row[field]
        results_table[field][letter_id] = fields_match(field, predicted_value, gold_value)

# Build the accuracy DataFrame: rows = fields, columns = L001 / L003 / L006 / accuracy
accuracy_df = pd.DataFrame(results_table).T  # transpose so fields are rows
accuracy_df = accuracy_df[gold_letter_ids]    # order columns L001, L003, L006
accuracy_df["accuracy"] = accuracy_df[gold_letter_ids].mean(axis=1)

print("Per-field extraction accuracy vs. GOLD:")
display(accuracy_df)

overall_accuracy = accuracy_df[gold_letter_ids].values.mean()
print(f"\nOverall accuracy across all fields and letters: {overall_accuracy:.2%}")

Per-field extraction accuracy vs. GOLD:


,L001,L003,L006,accuracy
applicant_name,True,True,True,1.0
amount_ghs,True,True,True,1.0
purpose,False,False,False,0.0
monthly_profit_ghs,True,True,True,1.0
has_collateral_or_guarantor,True,True,True,1.0
repayment_months,True,True,True,1.0



Overall accuracy across all fields and letters: 83.33%


### Part 4.2 — Reliability: is the system consistent?

In [22]:
# TODO: Run extract_fields() on letter L004 FIVE times at temperature=0 and FIVE times at
#   temperature=1.0.

# TODO: For each temperature, report how many of the 5 runs produced (a) valid JSON and
#   (b) identical values across runs. A simple approach: json.dumps(result, sort_keys=True)
#   and count unique strings.

# Run extract_fields on L004 five times at temperature=0 and five times at temperature=1.0.
# extract_fields() currently hardcodes temperature=0.0 inside it, so we define a small
# variant here that accepts a temperature argument, reusing your existing EXTRACT_PROMPT.

def extract_fields_at_temp(letter_text, temperature):
    system_prompt, user_prompt = EXTRACT_PROMPT(letter_text)
    try:
        llm_response = ask_llm(
            user_prompt=user_prompt,
            system_prompt=system_prompt,
            temperature=temperature,
        )
        response_content = llm_response.choices[0].message.content

        match = re.search(r'```json\n(.*?)```', response_content, re.DOTALL)
        json_string = match.group(1).strip() if match else response_content.strip()

        return json.loads(json_string)
    except json.JSONDecodeError as e:
        print(f"  Invalid JSON at temperature={temperature}: {e}")
        print(f"  Raw response: {response_content}")
        return None
    except Exception as e:
        print(f"  Unexpected error at temperature={temperature}: {e}")
        return None

letter_l004 = LETTERS["L004"]
n_runs = 5

reliability_results = {}

for temp in [0.0, 1.0]:
    print(f"\n--- Running L004 extraction 5x at temperature={temp} ---")
    outputs = []
    valid_json_count = 0

    for i in range(n_runs):
        result = extract_fields_at_temp(letter_l004, temperature=temp)
        if result is not None:
            valid_json_count += 1
            outputs.append(json.dumps(result, sort_keys=True))
        else:
            outputs.append(None)
        print(f"  Run {i+1}: {result}")

    valid_outputs = [o for o in outputs if o is not None]
    unique_outputs = set(valid_outputs)

    reliability_results[temp] = {
        "valid_json_count": valid_json_count,
        "unique_value_count": len(unique_outputs),
        "identical_runs": valid_json_count - len(unique_outputs) + (1 if unique_outputs else 0),
    }

    print(f"  Valid JSON: {valid_json_count}/{n_runs}")
    print(f"  Unique distinct outputs: {len(unique_outputs)} "
          f"(1 unique value = fully consistent, {n_runs} unique = fully inconsistent)")

reliability_df = pd.DataFrame(reliability_results).T
reliability_df.index.name = "temperature"
print("\nSummary:")
display(reliability_df)


--- Running L004 extraction 5x at temperature=0.0 ---
  Run 1: {'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'poultry farm', 'monthly_profit_ghs': 1500, 'has_collateral_or_guarantor': True, 'repayment_months': 18}
  Run 2: {'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'poultry farm', 'monthly_profit_ghs': 1500, 'has_collateral_or_guarantor': True, 'repayment_months': 18}
  Run 3: {'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'poultry farm', 'monthly_profit_ghs': 1500, 'has_collateral_or_guarantor': True, 'repayment_months': 18}
  Run 4: {'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'poultry farm', 'monthly_profit_ghs': 1500, 'has_collateral_or_guarantor': True, 'repayment_months': 18}
  Run 5: {'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'poultry farm', 'monthly_profit_ghs': 1500, 'has_collateral_or_guarantor': True, 'repayment_months': 18}
  Valid JSON: 5/5
  Unique distinct outputs: 1 (1 uniqu

,valid_json_count,unique_value_count,identical_runs
temperature,,,
0.0,5,1,5
1.0,5,2,4


### Part 4.3 — Hallucination probing

In [23]:
# TODO: Design TWO adversarial tests and run them:
#   Test 1 — Ask your summarizer a question about a detail that is NOT in a letter
#     (e.g. "What is the applicant's credit score?"). Does it admit the information is
#     absent, or does it invent one?
#   Test 2 — Feed your extractor an EMPTY or IRRELEVANT text (e.g. a weather report).
#     Does it return nulls, or does it fabricate an applicant?

# TODO: Record the outputs verbatim below and label each PASS or FAIL.

# Test 1: Ask the summarizer about a detail NOT present in a letter.
# Does it admit the information is absent, or invent an answer?

print("=== Test 1: Asking about missing information (credit score) ===")
probe_letter = LETTERS["L001"]  # no credit score mentioned anywhere
probe_question = (
    f"Based only on the following loan application letter, what is the applicant's "
    f"credit score?\n\nLetter:\n{probe_letter}"
)
probe_system_prompt = (
    "You are an assistant to a microfinance loan officer. Answer only using information "
    "explicitly stated in the letter. If the information is not present, say so clearly "
    "and do not guess or estimate."
)

test1_response = ask_llm(probe_question, system_prompt=probe_system_prompt, temperature=0.0)
test1_output = test1_response.choices[0].message.content
print(f"Model output:\n{test1_output}\n")

# TODO after reading the output: label PASS (admits absence) or FAIL (invents a number)
test1_verdict = "PASS"  # <-- change to "FAIL" if it invented a credit score
print(f"Verdict: {test1_verdict}")


# Test 2: Feed the extractor EMPTY or IRRELEVANT text (e.g. a weather report).
# Does it return nulls, or fabricate an applicant?

print("\n=== Test 2: Extracting from irrelevant text (weather report) ===")
irrelevant_text = (
    "Weather report for Accra: Today will be partly cloudy with a high of 31°C and a "
    "low of 24°C. Winds from the southwest at 12 km/h. Chance of rain in the evening. "
    "Humidity around 78%. Sunset expected at 6:12 PM."
)

test2_result = extract_fields(irrelevant_text)
print(f"Extracted fields:\n{test2_result}\n")

# TODO after reading the output: label PASS (all fields null/no fabricated applicant)
#   or FAIL (invented a name, amount, etc. out of the weather report)
test2_verdict = "PASS"  # <-- change to "FAIL" if it fabricated applicant data
print(f"Verdict: {test2_verdict}")

print("\n=== Summary ===")
print(f"Test 1 (missing detail probe): {test1_verdict}")
print(f"Test 2 (irrelevant text probe): {test2_verdict}")

=== Test 1: Asking about missing information (credit score) ===
Model output:
The applicant's credit score is not mentioned in the letter.

Verdict: PASS

=== Test 2: Extracting from irrelevant text (weather report) ===
Extracted fields:
{'applicant_name': None, 'amount_ghs': None, 'purpose': None, 'monthly_profit_ghs': None, 'has_collateral_or_guarantor': None, 'repayment_months': None}

Verdict: PASS

=== Summary ===
Test 1 (missing detail probe): PASS
Test 2 (irrelevant text probe): PASS


**Student Reasoning — Evaluation results**
*1. Report your extraction accuracy. Which field was hardest for the model and why?*
*2. What did the reliability experiment show about temperature and production systems?*
*3. Did your system hallucinate under probing? If yes, how could the prompt (or the system
design around it) reduce the risk?*

> **Answer:**
1. Overall accuracy accros all fields and all three gold letters was 83.33%. Broken down by various fields where all scored 100% except one field purpose which scored 0%. Purpose was the hardest field to score becuase fee text fields do not have one unambiguous correct string the way a number or boolean does, so an exact-match metric systematically under-scores otherwise-correct answers. For L001 it extracted "buy a deep freezer/expand into frozen foods" against a gold label of "buy deep freezer/expand into frozen foods".

2. It showed that for a letter with unambiguous, clearly-stated facts, the correct answer dominates the models's output distribution strongly enough that added randomness does not disturb it. For letters with more ambiguous or borderline phrasing, higher temperature would likely produce more variation.

3. No, both adversial tests passed. Both results were exactly what the explicit "if not stated, use null/d not guess" instructions in the prompts were designed to produce and they worked. However, two passig probes on two fairly obvious cases does not prove the system is hallucination-proof in general, it just shows the guardrail works for failure modes tested. To reduce further risk in production, add a code-level guardrail to check extracted string or number values that appear in or are direct;y derivable from the source letter before accepting them, rather than trusting the prompt instructions alone. And any output that fails such a check should be routed to mandatory human review instead of silently passing it downstream.

### Part 4.4 — Appropriateness: should this system exist?
No code in this part — just judgment, which is the scarcest skill in AI for business.

**Student Reasoning — Appropriateness**
*1. Letters L002 and L006 would likely be declined. If the bank fully automated decisions
with your system, who could be unfairly harmed, and how? Consider applicants who write
poorly in English but run solid businesses.*
*2. Loan letters contain personal data. What are the implications of sending them to a
third-party API in another country? What would you check before deploying this at a real
Ghanaian microfinance institution?*
*3. Name TWO concrete safeguards you would build around this system in production (think:
human review points, logging, appeal processes, monitoring).*

> **Answer:**
1. Applicants who write poorly in English but run genuinely solid businesses could be unfairly unharmed. Reading results from the purpose field, even a correct extraction can look off once measured by rigid criteria, and a model or a skimming human can easily conflate fluent, well-structured writing with creditworthiness.

2. Raises cross-border data protection questions like whether applicants have consensted to their data leaving the country, whether the provider retains or trains on submitted data, and how long logs are kept. Before deploying at a real Ghanaian microfinance institution, I wpuld check: the provider's data retention and commitments, whether cross-border transfer of financial data complies with the Data Protection Commission's requirements and the institution's own consent or disclosure processes.

3. a) Mandatory human sign-off before any adverse action. An officer must review the extracted fields against the original letter and read the brief before a decline is communicated to an applicant, especially because of the accuracy not being 100%.

b) Full audit logging and an appeal path. Logging every prompt, raw model response, and model used per application so any decision can be reconstructed later, paired with a formal process letting applicants request human re-review if they believe the AI-generated summary or extraction mispresented their application.

---
# Section 5 — Reflection

*Answer in a few sentences each:*

1. **Prompting as engineering:** How is iterating on a prompt similar to and different from
   iterating on the model hyperparameters you tuned in Lab 3?
2. **Trust:** After your Section 4 evaluation, would you trust this system to run unattended?
   What single evaluation result most influenced your answer?
3. **Cost and scale:** Estimate (from your `response.usage` numbers) the tokens needed to
   process 1,000 applications per month. What does that imply for provider choice?
4. **Looking back at the course:** You have now used classical ML (Lab 2), trained neural
   networks (Lab 3), and used a foundation model via API (Lab 4). For a task like this one,
   why does calling an API beat training your own model — and when would it not?

> **Answer:**
1. Hyperparameters tune numeric weights of a model through gradient descent, producing a fixed reproducible artifact once training finishes. Prompting changes no weights at all, there is no gradient to tell which direction to move, and there is no training cost, each iteration is a single API call, so feedback is essentially instant compared to retraining a network.

2. No, not fully unattended. The single most influential result is the 83.33% overall extraction accuracy, an error rate of rougly 1 in 6 fiels values is too high to let a system autnomously dtermine outcomes without a human checking its work.

3. Rougly 800,000-1,500,000 tokens per month. A free or low-cost, open-weight-backed provider is the practical choice for a task like this at MFI scale.

4. Calling an API beats training the model because no large labeled corpus of loan letters is needed, only three gold examples were used, purely for evaluating the API's output, not for training anything. It would not win when the task is narrow, extremely high-volume, and needs very low per-inference latency/cost at massive scale, or when there is a large proprietary labeled dataset and the task is really a structured prediction problem.

---
### Submission checklist

- [ ] All cells run top-to-bottom with no errors (`Kernel -> Restart & Run All`).
- [ ] **No API key anywhere in the notebook or the commit history.**
- [ ] Every **Student Reasoning** box is filled in with full sentences.
- [ ] `prompts.py` / `prompts.md` committed with your final prompt templates.
- [ ] Evaluation tables and adversarial test outputs visible in the saved notebook.
- [ ] Notebook pushed to `lab-4-llm-decision-support` with incremental commits.
- [ ] Repository link submitted to the course portal.
- [ ] AI Declaration form in Repository.